In [17]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import matplotlib.pyplot as plt

## Fama French 3

CAPM only explain return with 1 factor - market risk

CAPM
    R = Rf + B x MKT

FF3
    R = Rf = B1 x MKT + B2 x SMB + B3 x HML    

Fama French explain return with 3 factor
    1. Market Risk
    2. SMB  
        Small minus Big - Return of small stocks minus return of big stocks
    3. HML
        High Minus Low of Book Market Ratio
        Value Stocks - Growth Stocks


Newer model of Fama French added 2 more factor
    4. Profitability
    5. Investment    

In [18]:
sp500 = yf.download("SPY", period="5y")["Close"]["SPY"]
sp500_returns = sp500.pct_change(fill_method=None).dropna()

[*********************100%***********************]  1 of 1 completed


In [19]:
ff3 = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip"
ff3 = pd.read_csv(ff3, skiprows=3, index_col=0)
ff3 = ff3[:-1]

ff3.index = pd.to_datetime(ff3.index, format="%Y%m%d")
ff3 = ff3.astype(float) / 100

print(ff3.head())
print(ff3.tail())


            Mkt-RF     SMB     HML      RF
1926-07-01  0.0009 -0.0025 -0.0027  0.0001
1926-07-02  0.0045 -0.0033 -0.0006  0.0001
1926-07-06  0.0017  0.0030 -0.0039  0.0001
1926-07-07  0.0009 -0.0058  0.0002  0.0001
1926-07-08  0.0022 -0.0038  0.0019  0.0001
            Mkt-RF     SMB     HML      RF
2026-01-26  0.0040 -0.0074 -0.0023  0.0002
2026-01-27  0.0034 -0.0004  0.0022  0.0002
2026-01-28 -0.0010 -0.0105  0.0039  0.0002
2026-01-29 -0.0026 -0.0016  0.0139  0.0002
2026-01-30 -0.0055 -0.0054  0.0035  0.0002


In [20]:
ff3 = ff3.loc[sp500_returns.index[0]:sp500_returns.index[-1]]
sp500_returns = sp500_returns.loc[ff3.index]

excess_return = sp500_returns.values - ff3["RF"].values

In [22]:
X = ff3[["Mkt-RF", "SMB", "HML"]]
X = sm.add_constant(X)

model = sm.OLS(excess_return, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.993
Model:                            OLS   Adj. R-squared:                  0.993
Method:                 Least Squares   F-statistic:                 5.901e+04
Date:                Sun, 22 Mar 2026   Prob (F-statistic):               0.00
Time:                        13:26:41   Log-Likelihood:                 6846.8
No. Observations:                1221   AIC:                        -1.369e+04
Df Residuals:                    1217   BIC:                        -1.367e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       4.954e-06   2.55e-05      0.194      0.8

In [24]:
tsla = yf.download("TSLA", period="5y")["Close"]["TSLA"]
tsla_returns = tsla.pct_change(fill_method=None).dropna()

excess_returns_tsla = tsla_returns.loc[ff3.index]

model = sm.OLS(excess_returns_tsla, X).fit()
print(model.summary())

[*********************100%***********************]  1 of 1 completed

                            OLS Regression Results                            
Dep. Variable:                   TSLA   R-squared:                       0.373
Model:                            OLS   Adj. R-squared:                  0.371
Method:                 Least Squares   F-statistic:                     241.0
Date:                Sun, 22 Mar 2026   Prob (F-statistic):          1.04e-122
Time:                        13:54:25   Log-Likelihood:                 2558.6
No. Observations:                1221   AIC:                            -5109.
Df Residuals:                    1217   BIC:                            -5089.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0008      0.001      0.939      0.3